# PaddleOCR vs. Tesseract: Accuracy on Government ID Documents

This notebook benchmarks **PaddleOCR** against **Tesseract (`pytesseract`)** on scanned/photographed government ID documents (passports, driving licences, ID cards).

**Dataset: [MIDV-500](https://arxiv.org/abs/1807.05786)** (Smart Engines / Recognition and Perception Systems Lab).
- 500 video clips of **50 identity document types** (17 ID cards, 14 passports, 13 driving licences, 6 other).
- Documents are **mock IDs** created specifically for research (fictional people / consenting volunteers) — no real PII, freely redistributable for research use.
- Ships with per-frame **ground-truth document quadrangles** (the 4 corner points of the document in each photo, for cropping/rectifying it) and, per document type, a **ground-truth JSON of the actual text field values** (name, document number, dates, etc.) — since each type is one physical mock document filmed 10 times, one template of field text covers every frame of that type. This is what we score OCR output against.
- It's split into 50 small per-type zip files, so we only download a handful of types instead of the full dataset. Each type's zip still bundles the *source videos* the frames were extracted from (that's most of the download size, ~500 MB-1 GB per type) — we delete them right after extracting since we only need the frame images.

We picked 4 Latin-script document types spanning the categories you care about:
| Code | Country | Document type |
|---|---|---|
| `01_alb_id` | Albania | National ID card |
| `12_deu_drvlic_new` | Germany | Driving licence |
| `05_aze_passport` | Azerbaijan | Passport |
| `48_usa_passportcard` | USA | Passport card |

Feel free to swap these for other codes from the [full list of 50](https://github.com/fcakyon/midv500/blob/master/midv500/download_dataset.py) — just keep in mind PaddleOCR/Tesseract need a language model that matches the document's script.

## 1. Install dependencies
Run this once per Colab session (takes ~2-3 minutes, mostly for PaddleOCR).

In [ ]:
!apt-get -qq update && apt-get -qq install -y tesseract-ocr
!pip -q install pytesseract paddleocr paddlepaddle rapidfuzz opencv-python-headless pandas matplotlib tqdm

## 2. Download a small slice of MIDV-500
We reuse the download/unzip helpers from the [`midv500`](https://pypi.org/project/midv500/) PyPI package but call them ourselves on just the 4 zip files we picked, instead of the whole 50-type dataset.

In [ ]:
!pip -q install midv500

import os
import shutil
from midv500.utils import download, unzip

DATA_DIR = "midv500_data"
os.makedirs(DATA_DIR, exist_ok=True)

DOC_TYPES = [
    "01_alb_id",
    "12_deu_drvlic_new",
    "05_aze_passport",
    "48_usa_passportcard",
]

BASE_URL = "ftp://smartengines.com/midv-500/dataset"

for doc_type in DOC_TYPES:
    zip_path = os.path.join(DATA_DIR, f"{doc_type}.zip")
    extract_marker = os.path.join(DATA_DIR, doc_type)
    if os.path.exists(extract_marker):
        print(f"{doc_type}: already extracted, skipping")
        continue
    url = f"{BASE_URL}/{doc_type}.zip"
    print(f"Downloading {url} ...")
    download(url, DATA_DIR)
    unzip(zip_path, DATA_DIR)
    os.remove(zip_path)
    # we only need images/ + ground_truth/; the bundled source videos are most of the disk usage
    videos_dir = os.path.join(extract_marker, "videos")
    if os.path.isdir(videos_dir):
        shutil.rmtree(videos_dir)
    print(f"{doc_type}: done")

If the FTP download times out or is blocked on your network, fall back to the full-dataset pip helper instead (bigger download, ~5-6 GB):
```python
import midv500
midv500.download_dataset("midv500_data", "midv500")
```

## 3. Explore the extracted structure
MIDV-500's ground truth per document type has two parts:
- **`<doc_type>/ground_truth/<clip>/<frame>.json`** — per-frame `"quad"`: the 4 corner points of the document in that frame (for detection/cropping).
- **`<doc_type>/ground_truth/<doc_type>.json`** — a single template file (sitting *inside* `ground_truth/`, alongside the per-clip folders, not at the type's root) with the actual **text field values** printed on that document. Since all clips of a type show the same physical document, one template covers every frame.

Let's confirm the schema before writing generic code against it.

In [ ]:
import glob, json

sample_type = DOC_TYPES[0]
root = os.path.join(DATA_DIR, sample_type)
gt_root = os.path.join(root, "ground_truth")

print("Top-level contents:", os.listdir(root))
print("ground_truth/ contents:", os.listdir(gt_root))

clip_dirs = sorted(d for d in os.listdir(gt_root) if os.path.isdir(os.path.join(gt_root, d)))
clip_dir = os.path.join(gt_root, clip_dirs[0])
sample_gt = sorted(os.listdir(clip_dir))[0]
with open(os.path.join(clip_dir, sample_gt)) as f:
    print(f"\nSample per-frame ground truth ({clip_dirs[0]}/{sample_gt}):")
    print(json.dumps(json.load(f), indent=2))

template_path = os.path.join(gt_root, f"{sample_type}.json")
with open(template_path, encoding="utf-8") as f:
    template_gt = json.load(f)
print(f"\nTemplate (field-text) ground truth ({template_path}):")
print(json.dumps(template_gt, indent=2, ensure_ascii=False)[:2000])

## 4. Helper functions
`extract_gt_strings` walks the template ground-truth JSON recursively and pulls out every text value that looks like a real field (skips numeric coordinates, single characters, etc.), so it works regardless of small schema differences between document types instead of hard-coding field names.

In [ ]:
import re


def find_template_gt(doc_type: str):
    """Load the per-document-type ground truth JSON (field text values)."""
    path = os.path.join(DATA_DIR, doc_type, "ground_truth", f"{doc_type}.json")
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def extract_gt_strings(obj, out=None):
    """Recursively collect plausible text field values out of a (nested) GT json."""
    if out is None:
        out = []
    if isinstance(obj, dict):
        for v in obj.values():
            extract_gt_strings(v, out)
    elif isinstance(obj, list):
        for v in obj:
            extract_gt_strings(v, out)
    elif isinstance(obj, str):
        s = obj.strip()
        # skip empty / pure punctuation / single characters
        if len(s) >= 2 and re.search(r"[A-Za-z0-9]", s):
            out.append(s)
    return out


def find_frames(doc_type: str, max_clips=2, max_frames_per_clip=5):
    """Return a small sample of (image_path, quad_json_path) pairs for a doc type."""
    root = os.path.join(DATA_DIR, doc_type)
    images_root = os.path.join(root, "images")
    gt_root = os.path.join(root, "ground_truth")
    pairs = []
    # images/ and ground_truth/ each also contain a stray <doc_type> file
    # (a reference scan / the template ground truth) alongside the clip folders
    clip_dirs = sorted(d for d in os.listdir(images_root) if os.path.isdir(os.path.join(images_root, d)))[:max_clips]
    for clip in clip_dirs:
        img_dir = os.path.join(images_root, clip)
        frame_files = sorted(os.listdir(img_dir))[:max_frames_per_clip]
        for fname in frame_files:
            img_path = os.path.join(img_dir, fname)
            stem = os.path.splitext(fname)[0]
            quad_path = os.path.join(gt_root, clip, stem + ".json")
            if os.path.exists(quad_path):
                pairs.append((img_path, quad_path))
    return pairs

## 5. Rectify the document from each frame
MIDV-500 frames are photos of the document lying on a background, not clean scans. We use the ground-truth quadrangle to perspective-warp each frame into a straightened, cropped image of just the document — this is what you'd realistically feed an OCR engine in a document-processing pipeline.

In [ ]:
import cv2
import numpy as np

def order_points(pts):
    pts = np.array(pts, dtype="float32")
    s = pts.sum(axis=1)
    diff = np.diff(pts, axis=1).flatten()
    ordered = np.zeros((4, 2), dtype="float32")
    ordered[0] = pts[np.argmin(s)]       # top-left
    ordered[2] = pts[np.argmax(s)]       # bottom-right
    ordered[1] = pts[np.argmin(diff)]    # top-right
    ordered[3] = pts[np.argmax(diff)]    # bottom-left
    return ordered


def rectify_document(image_path, quad_path, out_width=1000):
    img = cv2.imread(image_path)
    with open(quad_path) as f:
        quad = json.load(f)["quad"]
    src = order_points(quad)
    (tl, tr, br, bl) = src
    w = max(np.linalg.norm(br - bl), np.linalg.norm(tr - tl))
    h = max(np.linalg.norm(tr - br), np.linalg.norm(tl - bl))
    aspect = h / w if w > 0 else 0.63
    out_h = int(out_width * aspect)
    dst = np.array([[0, 0], [out_width - 1, 0], [out_width - 1, out_h - 1], [0, out_h - 1]], dtype="float32")
    M = cv2.getPerspectiveTransform(src, dst)
    warped = cv2.warpPerspective(img, M, (out_width, out_h))
    return warped

Optional sanity check — see what we're actually about to feed the OCR engines:

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, len(DOC_TYPES), figsize=(6 * len(DOC_TYPES), 6))
for ax, doc_type in zip(axes, DOC_TYPES):
    img_path, quad_path = find_frames(doc_type, max_clips=1, max_frames_per_clip=1)[0]
    warped = rectify_document(img_path, quad_path)
    ax.imshow(cv2.cvtColor(warped, cv2.COLOR_BGR2RGB))
    ax.set_title(doc_type)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 6. Run all three OCR engines
Alongside PaddleOCR's default (server) models and Tesseract, we also benchmark **PaddleOCR's mobile models** (`PP-OCRv5_mobile_det` / `PP-OCRv5_mobile_rec`) — the lightweight detection/recognition models PaddleOCR ships for on-device/edge deployment (a few MB each, vs. 100MB+ for the server models). If PaddleOCR's accuracy is winning but its latency is the blocker for an on-device use case, this is the variant to actually deploy — it trades some accuracy for a large speed win instead of Tesseract's much larger accuracy drop.

In [ ]:
import pytesseract
from paddleocr import PaddleOCR

# PaddleOCR 3.x renamed use_angle_cls -> use_textline_orientation and dropped show_log;
# .ocr() now returns dict-like OCRResult objects instead of the old list-of-tuples format.
# enable_mkldnn=False works around a NotImplementedError some CPU/oneDNN builds hit
# (ConvertPirAttribute2RuntimeAttribute) during inference; harmless if you're on GPU.
# The 3.x default pipeline also runs doc-orientation-classification and doc-unwarping
# models before detection/recognition; we already rectify the document ourselves, so
# disabling both cuts a lot of redundant per-image latency on CPU.
paddle_engine = PaddleOCR(
    use_textline_orientation=True,
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    lang="en",
    enable_mkldnn=False,
)

# Same pipeline, but forced onto the PP-OCRv5 *mobile* det/rec models instead of the
# (default) server models - this is the lightweight pair actually meant for on-device/edge
# deployment (Android/iOS via Paddle Lite, or exported to ONNX/TFLite). Downloads the mobile
# model weights (a few MB) on first use.
paddle_engine_mobile = PaddleOCR(
    text_detection_model_name="PP-OCRv5_mobile_det",
    text_recognition_model_name="PP-OCRv5_mobile_rec",
    use_textline_orientation=True,
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    lang="en",
    enable_mkldnn=False,
)

def ocr_tesseract(img_bgr):
    rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    return pytesseract.image_to_string(rgb)

def ocr_paddle(img_bgr):
    result = paddle_engine.predict(img_bgr)
    if not result:
        return ""
    return "\n".join(result[0]["rec_texts"])

def ocr_paddle_mobile(img_bgr):
    result = paddle_engine_mobile.predict(img_bgr)
    if not result:
        return ""
    return "\n".join(result[0]["rec_texts"])

## 7. Scoring: field recall + character error rate
For each document we know the ground-truth field values (names, numbers, dates, ...) but not their exact pixel location, so we score OCR output two ways:
1. **Field recall** — for each ground-truth field value, fuzzy-search for it inside the engine's raw output (`rapidfuzz.partial_ratio`). Fraction of fields found = recall for that image.
2. **Character Error Rate (CER)** — Levenshtein distance between the normalized OCR text and the normalized concatenation of all ground-truth fields, divided by the ground-truth length. Lower is better.

In [ ]:
from rapidfuzz import fuzz
from rapidfuzz.distance import Levenshtein

MATCH_THRESHOLD = 85  # 0-100, rapidfuzz partial_ratio

def normalize(text):
    return re.sub(r"\s+", " ", text).strip().lower()

def field_recall(ocr_text, gt_fields):
    norm_ocr = normalize(ocr_text)
    if not gt_fields:
        return None
    hits = sum(1 for field in gt_fields if fuzz.partial_ratio(normalize(field), norm_ocr) >= MATCH_THRESHOLD)
    return hits / len(gt_fields)

def character_error_rate(ocr_text, gt_fields):
    ref = normalize(" ".join(gt_fields))
    hyp = normalize(ocr_text)
    if not ref:
        return None
    return Levenshtein.distance(hyp, ref) / len(ref)

## 8. Run the benchmark

In [ ]:
import time
import pandas as pd
from tqdm.auto import tqdm

MAX_CLIPS_PER_TYPE = 2
MAX_FRAMES_PER_CLIP = 5

rows = []

for doc_type in DOC_TYPES:
    gt_fields = extract_gt_strings(find_template_gt(doc_type))
    frames = find_frames(doc_type, MAX_CLIPS_PER_TYPE, MAX_FRAMES_PER_CLIP)
    print(f"{doc_type}: {len(frames)} sample frames, {len(gt_fields)} ground-truth field strings")

    for img_path, quad_path in tqdm(frames, desc=doc_type):
        try:
            doc_img = rectify_document(img_path, quad_path)
        except Exception as e:
            print(f"skip {img_path}: {e}")
            continue

        for engine_name, ocr_fn in [("tesseract", ocr_tesseract), ("paddleocr", ocr_paddle), ("paddleocr_mobile", ocr_paddle_mobile)]:
            start = time.time()
            text = ocr_fn(doc_img)
            elapsed = time.time() - start
            rows.append({
                "doc_type": doc_type,
                "image": os.path.basename(img_path),
                "image_path": img_path,
                "quad_path": quad_path,
                "engine": engine_name,
                "seconds": elapsed,
                "text": text,
                "field_recall": field_recall(text, gt_fields),
                "cer": character_error_rate(text, gt_fields),
                "chars_extracted": len(text.strip()),
            })

results = pd.DataFrame(rows)
results.head()

## 9. Results

In [ ]:
summary = results.groupby(["doc_type", "engine"])[["field_recall", "cer", "seconds"]].mean().round(3)
summary

In [ ]:
overall = results.groupby("engine")[["field_recall", "cer", "seconds"]].mean().round(3)
overall

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
metrics = ["field_recall", "cer", "seconds"]
titles = ["Field recall (higher=better)", "Character error rate (lower=better)", "Avg. seconds/image"]
engine_colors = ["#1f77b4", "#ff7f0e", "#2ca02c"]

for ax, metric, title in zip(axes, metrics, titles):
    colors = [engine_colors[i % len(engine_colors)] for i in range(len(overall))]
    overall[metric].plot(kind="bar", ax=ax, color=colors)
    ax.set_title(title)
    ax.set_xlabel("")

plt.tight_layout()
plt.show()

## 10. Additional metrics: token precision/recall/F1, word error rate, exact match
`field_recall` (fuzzy, whole-field) and `cer` (character-level) are already computed. These add complementary views:
- **Token precision** — of the words the engine outputs, what fraction also appear in a ground-truth field? Low precision usually means the engine is reading extra boilerplate/noise (or hallucinating characters), not necessarily that it got the *fields* wrong.
- **Token recall** — same idea as `field_recall` but at the word level instead of whole-field-string level; more forgiving of an engine that reads a field's words correctly but splits/merges them differently.
- **F1** — harmonic mean of the two.
- **Word error rate (WER)** — Levenshtein distance at the word level, normalized by ground-truth word count. Lower is better; unlike CER it isn't dominated by long correctly-read fields swamping a few character typos.
- **Exact match rate** — fraction of ground-truth fields that appear byte-for-byte (after normalization) in the OCR output — a stricter cousin of `field_recall`'s fuzzy threshold.

**Caveat:** our ground truth only lists specific fields (name, number, dates, ...), not a full-page transcription — so precision slightly *understates* real accuracy, since correctly-read text that isn't one of our tracked fields (e.g. "REPUBLIC OF ALBANIA") still counts as a false positive here.

In [ ]:
gt_fields_by_type = {dt: extract_gt_strings(find_template_gt(dt)) for dt in DOC_TYPES}

def tokenize(text):
    return re.findall(r"[a-z0-9]+", text.lower())

def token_prf1(ocr_text, gt_fields):
    gt_tokens = set()
    for field in gt_fields:
        gt_tokens.update(tokenize(field))
    ocr_tokens = tokenize(ocr_text)
    if not gt_tokens or not ocr_tokens:
        return None, None, None
    ocr_token_set = set(ocr_tokens)
    matched = gt_tokens & ocr_token_set
    precision = len(matched) / len(ocr_token_set)
    recall = len(matched) / len(gt_tokens)
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    return precision, recall, f1

def word_error_rate(ocr_text, gt_fields):
    ref = normalize(" ".join(gt_fields)).split()
    hyp = normalize(ocr_text).split()
    if not ref:
        return None
    return Levenshtein.distance(hyp, ref) / len(ref)

def exact_match_rate(ocr_text, gt_fields):
    norm_ocr = normalize(ocr_text)
    if not gt_fields:
        return None
    hits = sum(1 for field in gt_fields if normalize(field) in norm_ocr)
    return hits / len(gt_fields)

extra_rows = []
for _, row in results.iterrows():
    gt_fields = gt_fields_by_type[row["doc_type"]]
    precision, recall_tok, f1 = token_prf1(row["text"], gt_fields)
    extra_rows.append({
        "precision": precision,
        "recall_tok": recall_tok,
        "f1": f1,
        "wer": word_error_rate(row["text"], gt_fields),
        "exact_match": exact_match_rate(row["text"], gt_fields),
    })

results = pd.concat([results.reset_index(drop=True), pd.DataFrame(extra_rows)], axis=1)

extra_summary = results.groupby("engine")[["precision", "recall_tok", "f1", "wer", "exact_match"]].mean().round(3)
extra_summary

## 11. Look at a random example
Pick a random image, show the original photographed frame (with the ground-truth quadrangle outlined) next to the rectified crop actually fed to OCR, the ground-truth fields, and what each engine read — the fastest way to sanity-check whether the numbers above match what you'd expect by eye. Re-run the cell to see a different one.

In [ ]:
import random

doc_type, image_name = random.choice(list(results.groupby(["doc_type", "image"]).groups.keys()))
sub = results[(results["doc_type"] == doc_type) & (results["image"] == image_name)]
row0 = sub.iloc[0]

original = cv2.imread(row0["image_path"])
with open(row0["quad_path"]) as f:
    quad = json.load(f)["quad"]
quad_pts = np.array(quad, dtype=np.int32).reshape((-1, 1, 2))
annotated = original.copy()
cv2.polylines(annotated, [quad_pts], isClosed=True, color=(0, 0, 255), thickness=4)

warped = rectify_document(row0["image_path"], row0["quad_path"])

fig, axes = plt.subplots(1, 2, figsize=(14, 8))
axes[0].imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
axes[0].set_title(f"Original frame\n{doc_type} / {image_name}")
axes[0].axis("off")
axes[1].imshow(cv2.cvtColor(warped, cv2.COLOR_BGR2RGB))
axes[1].set_title("Rectified crop (fed to OCR)")
axes[1].axis("off")
plt.tight_layout()
plt.show()

print("Ground-truth fields:")
for field in gt_fields_by_type[doc_type]:
    print(" -", field)

for _, r in sub.iterrows():
    print(f"\n--- {r['engine']} output (field_recall={r['field_recall']:.2f}, cer={r['cer']:.2f}) ---")
    print(r["text"])

## 12. Try your own photo
Upload a photo of an ID and run both engines on it. There's no ground truth for your own photo, so this is a qualitative side-by-side rather than a scored comparison — read both outputs and judge for yourself.

Unlike MIDV-500, we don't have pre-annotated corners for your photo, so we detect the document edges ourselves with a classic edge-detection + largest-rectangle-contour approach, then perspective-warp it exactly like section 5 did. This is far less robust than MIDV-500's curated ground truth — it works best when the ID is photographed roughly straight-on against a plain, contrasting background. If it fails to find a clean rectangle, it falls back to running OCR on the full photo.

In [ ]:
from google.colab import files

uploaded = files.upload()
user_image_path = next(iter(uploaded))

In [ ]:
def detect_document_quad(img):
    """Classic document-scanner approach: edges -> contours -> largest 4-point shape."""
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    edged = cv2.Canny(blurred, 50, 200)
    edged = cv2.dilate(edged, None, iterations=2)
    edged = cv2.erode(edged, None, iterations=1)

    contours, _ = cv2.findContours(edged, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)
    contours = sorted(contours, key=cv2.contourArea, reverse=True)[:10]

    for c in contours:
        peri = cv2.arcLength(c, True)
        approx = cv2.approxPolyDP(c, 0.02 * peri, True)
        if len(approx) == 4 and cv2.contourArea(approx) > 0.1 * img.shape[0] * img.shape[1]:
            return approx.reshape(4, 2).astype("float32")
    return None


def warp_quad(img, quad, out_width=1000):
    """Same perspective-warp math as rectify_document, but for an in-memory quad."""
    src = order_points(quad)
    (tl, tr, br, bl) = src
    w = max(np.linalg.norm(br - bl), np.linalg.norm(tr - tl))
    h = max(np.linalg.norm(tr - br), np.linalg.norm(tl - bl))
    aspect = h / w if w > 0 else 0.63
    out_h = int(out_width * aspect)
    dst = np.array([[0, 0], [out_width - 1, 0], [out_width - 1, out_h - 1], [0, out_h - 1]], dtype="float32")
    M = cv2.getPerspectiveTransform(src, dst)
    return cv2.warpPerspective(img, M, (out_width, out_h))


user_img = cv2.imread(user_image_path)

# keep runtime reasonable on large phone photos
h, w = user_img.shape[:2]
scale = 1500 / max(h, w)
if scale < 1:
    user_img = cv2.resize(user_img, (int(w * scale), int(h * scale)))

quad = detect_document_quad(user_img)

if quad is not None:
    annotated = user_img.copy()
    cv2.polylines(annotated, [quad.astype(np.int32).reshape((-1, 1, 2))], isClosed=True, color=(0, 0, 255), thickness=4)
    user_doc_img = warp_quad(user_img, quad)
else:
    print("Could not automatically detect the document edges - using the full photo instead.")
    print("For best results, retake/crop so the ID fills most of the frame against a plain background.")
    annotated = user_img
    user_doc_img = user_img

fig, axes = plt.subplots(1, 2, figsize=(14, 8))
axes[0].imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
axes[0].set_title("Detected document edges" if quad is not None else "Original (no edges detected)")
axes[0].axis("off")
axes[1].imshow(cv2.cvtColor(user_doc_img, cv2.COLOR_BGR2RGB))
axes[1].set_title("Rectified crop (fed to OCR)")
axes[1].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
for engine_name, ocr_fn in [("tesseract", ocr_tesseract), ("paddleocr", ocr_paddle), ("paddleocr_mobile", ocr_paddle_mobile)]:
    start = time.time()
    text = ocr_fn(user_doc_img)
    elapsed = time.time() - start
    print(f"\n=== {engine_name} ({elapsed:.1f}s) ===")
    print(text)

## Notes & next steps
- **Field recall** is a proxy for real-world extraction accuracy ("did the engine actually read the ID number / name / date correctly"), which matters more than raw character accuracy for ID-processing use cases.
- **`paddleocr_mobile`** (PP-OCRv5 mobile det/rec) is the variant worth deploying if you need this to run on-device (phone/edge) rather than a server — compare its `seconds` and `field_recall` against `paddleocr` (server models) above to see the actual speed/accuracy trade you're making. For real mobile deployment you'd additionally export these models to Paddle Lite / ONNX / TFLite rather than running the Python inference engine on-device.
- Try `lang="en"` vs. other PaddleOCR language packs (e.g. `"latin"`, `"german"`) — accuracy is very sensitive to the language model matching the document's script/diacritics.
- Try different Tesseract page-segmentation modes (`pytesseract.image_to_string(img, config="--psm 6")`) — ID cards/passports are often single uniform text blocks, which PSM 6 handles better than the default.
- To go further, MIDV-500 also has the harder **MIDV-2019** variant (same documents, but with motion blur / low light / projective distortion) if you want to stress-test both engines.
- Swap `DOC_TYPES` to explore other scripts (Cyrillic, CJK, Arabic) — that's where the two engines' accuracy gaps tend to widen the most.